# 🚀 AI Squad Internship — Week 1: Prompt Stress-Test Suite

**Goal:** Systematically compare prompt strategies and LLM models across 3 real business tasks.

| | |
|---|---|
| **Tasks** | Log Triage Summarizer · Arabic PII Redaction · Complaint Reply Drafter |
| **Strategies** | Zero-Shot · Few-Shot · Chain-of-Thought |
| **Models** | GPT-4o-mini (OpenAI) · Gemini 2.0 Flash (Google) |
| **Total runs** | 3 × 3 × 3 × 2 = **54 API calls** |

## 0. Setup & Imports

In [ ]:
import os, sys, json, time, yaml
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import display, Markdown
from dotenv import load_dotenv

# Add project root to path so src.* imports work
sys.path.insert(0, os.path.abspath('..'))

load_dotenv('../.env')

matplotlib.rcParams['figure.dpi'] = 120
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 60)

print('✅ Environment loaded')
print(f"  OPENAI_API_KEY: {'✅ Set' if os.getenv('OPENAI_API_KEY') else '❌ Missing'}")
print(f"  GEMINI_API_KEY: {'✅ Set' if os.getenv('GEMINI_API_KEY') else '❌ Missing'}")

## 1. Load Configuration & Inputs

In [ ]:
# Load config
with open('../config/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Load inputs
with open('../data/inputs.json', 'r', encoding='utf-8') as f:
    inputs = json.load(f)

tasks = ['task1', 'task2', 'task3']
strategies = ['zero_shot', 'few_shot', 'chain_of_thought']

print('📋 Configuration loaded:')
for model_key, model_cfg in config['models'].items():
    print(f"  {model_key}: {model_cfg['model_name']} ({model_cfg['provider']})")

print(f'\n📂 Inputs loaded:')
for task, task_inputs in inputs.items():
    print(f"  {task}: {len(task_inputs)} inputs")

## 2. Initialize LLM Clients

In [ ]:
import openai
from google import genai
from google.genai import types

openai_client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
gemini_client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

# Config shortcuts
gpt_cfg = config['models']['gpt_4o_mini']
gem_cfg = config['models']['gemini_2_0_flash']

def call_gpt(prompt: str) -> dict:
    """Call GPT-4o-mini and return {content, prompt_tokens, completion_tokens, latency}"""
    try:
        t0 = time.perf_counter()
        resp = openai_client.chat.completions.create(
            model=gpt_cfg['model_name'],
            messages=[{'role': 'user', 'content': prompt}],
            temperature=gpt_cfg['temperature'],
            max_tokens=gpt_cfg['max_tokens']
        )
        latency = time.perf_counter() - t0
        return {
            'content': resp.choices[0].message.content,
            'prompt_tokens': resp.usage.prompt_tokens,
            'completion_tokens': resp.usage.completion_tokens,
            'latency': latency
        }
    except Exception as e:
        return {'content': f'Error: {e}', 'prompt_tokens': 0, 'completion_tokens': 0, 'latency': 0}

def call_gemini(prompt: str) -> dict:
    """Call Gemini 2.0 Flash and return {content, prompt_tokens, completion_tokens, latency}"""
    try:
        t0 = time.perf_counter()
        resp = gemini_client.models.generate_content(
            model=gem_cfg['model_name'],
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=gem_cfg['temperature'],
                max_output_tokens=gem_cfg['max_tokens']
            )
        )
        latency = time.perf_counter() - t0
        p_tok = resp.usage_metadata.prompt_token_count if resp.usage_metadata else None
        c_tok = resp.usage_metadata.candidates_token_count if resp.usage_metadata else None
        return {
            'content': resp.text,
            'prompt_tokens': p_tok,
            'completion_tokens': c_tok,
            'latency': latency
        }
    except Exception as e:
        return {'content': f'Error: {e}', 'prompt_tokens': 0, 'completion_tokens': 0, 'latency': 0}

print('✅ LLM clients initialized')

## 3. Helper Functions

In [ ]:
import tiktoken

def load_prompt(task: str, strategy: str) -> str:
    path = f'../prompts/{task}/{strategy}.txt'
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

def count_tokens(text: str) -> int:
    enc = tiktoken.get_encoding('cl100k_base')
    return len(enc.encode(text))

def calc_cost(p_tokens, c_tokens, cost_p, cost_c):
    return (p_tokens / 1_000_000) * cost_p + (c_tokens / 1_000_000) * cost_c

def validate_json(text: str) -> bool:
    try:
        clean = text.replace('```json', '').replace('```', '').strip()
        json.loads(clean)
        return True
    except:
        return False

print('✅ Helper functions ready')

## 4. Run the Full Evaluation Matrix

> ⚠️ This runs **54 API calls** — takes ~2–4 minutes depending on network speed.

In [ ]:
results = []
total = len(tasks) * len(strategies) * 3 * 2
done = 0

models_map = {
    'gpt_4o_mini':     {'fn': call_gpt,    'cfg': gpt_cfg, 'provider': 'openai'},
    'gemini_2_0_flash': {'fn': call_gemini, 'cfg': gem_cfg, 'provider': 'gemini'},
}

for task in tasks:
    print(f'\n{'='*50}\n📌 {task.upper()}')
    task_inputs = inputs[task]
    for strategy in strategies:
        prompt_template = load_prompt(task, strategy)
        for i, input_text in enumerate(task_inputs):
            formatted_prompt = prompt_template.format(input_text=input_text)
            for model_key, model_info in models_map.items():
                done += 1
                print(f'  [{done:02d}/{total}] {strategy} | input {i+1} | {model_key}', end=' ... ')
                
                resp = model_info['fn'](formatted_prompt)
                
                p_tok = resp['prompt_tokens'] or count_tokens(formatted_prompt)
                c_tok = resp['completion_tokens'] or count_tokens(resp['content'])
                cost  = calc_cost(p_tok, c_tok,
                                  model_info['cfg']['cost_per_1m_prompt_tokens'],
                                  model_info['cfg']['cost_per_1m_completion_tokens'])
                is_valid_json = validate_json(resp['content']) if task == 'task1' else None
                
                results.append({
                    'Task':                task,
                    'Strategy':            strategy,
                    'Input_Index':         i + 1,
                    'Model':               model_key,
                    'Provider':            model_info['provider'],
                    'Latency_sec':         round(resp['latency'], 4),
                    'Prompt_Tokens':       p_tok,
                    'Completion_Tokens':   c_tok,
                    'Total_Tokens':        p_tok + c_tok,
                    'Estimated_Cost_USD':  round(cost, 6),
                    'Output_Length':       len(resp['content']),
                    'Valid_JSON':          is_valid_json,
                    'Response':            resp['content']
                })
                
                status = '✅ JSON' if is_valid_json else ('⚠️ no JSON' if task == 'task1' else '✅')
                print(f'{resp["latency"]:.2f}s | {p_tok+c_tok} tokens | ${cost:.6f} | {status}')

df = pd.DataFrame(results)
print(f'\n✅ Done! {len(df)} rows collected.')

## 5. Results Overview

In [ ]:
print('📊 Full Results DataFrame (first 10 rows):')
display(df[['Task','Strategy','Input_Index','Model','Latency_sec','Total_Tokens','Estimated_Cost_USD','Valid_JSON']].head(10))

In [ ]:
print('📈 Summary by Model:')
summary = df.groupby('Model').agg(
    Avg_Latency=('Latency_sec', 'mean'),
    Total_Cost_USD=('Estimated_Cost_USD', 'sum'),
    Total_Tokens=('Total_Tokens', 'sum'),
    Avg_Output_Len=('Output_Length', 'mean')
).round(4).reset_index()
display(summary)

In [ ]:
# Task 1 JSON reliability
task1_df = df[df['Task'] == 'task1']
if not task1_df.empty:
    print('🔍 Task 1 — JSON Output Validity (%):')    
    json_rate = task1_df.groupby(['Model', 'Strategy'])['Valid_JSON'].mean() * 100
    display(json_rate.reset_index().rename(columns={'Valid_JSON': 'JSON_Success_%'}))

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison — Prompt Stress-Test Suite', fontsize=14, fontweight='bold')

colors = ['#4361ee', '#f72585']

# Cost comparison
cost_agg = df.groupby('Model')['Estimated_Cost_USD'].sum()
cost_agg.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Total Cost (USD)', fontweight='bold')
axes[0].set_ylabel('Cost (USD)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=15, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Latency comparison
latency_agg = df.groupby('Model')['Latency_sec'].mean()
latency_agg.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white', width=0.5)
axes[1].set_title('Avg Latency (sec)', fontweight='bold')
axes[1].set_ylabel('Seconds')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=15, ha='right')
axes[1].grid(axis='y', alpha=0.3)

# Latency by strategy
lat_strat = df.groupby(['Strategy', 'Model'])['Latency_sec'].mean().unstack()
lat_strat.plot(kind='bar', ax=axes[2], color=colors, edgecolor='white', width=0.6)
axes[2].set_title('Avg Latency by Strategy', fontweight='bold')
axes[2].set_ylabel('Seconds')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=15, ha='right')
axes[2].grid(axis='y', alpha=0.3)
axes[2].legend(fontsize=8)

plt.tight_layout()
os.makedirs('../outputs/charts', exist_ok=True)
plt.savefig('../outputs/charts/notebook_comparison.png', bbox_inches='tight')
plt.show()
print('✅ Chart saved to outputs/charts/notebook_comparison.png')

In [ ]:
# Per-task latency heatmap
pivot = df.groupby(['Task', 'Model'])['Latency_sec'].mean().unstack()

fig, ax = plt.subplots(figsize=(8, 3))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=15, ha='right')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('Avg Latency Heatmap (sec) — Task × Model', fontweight='bold')
plt.colorbar(im, ax=ax, label='Latency (s)')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.2f}', ha='center', va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/charts/latency_heatmap.png', bbox_inches='tight')
plt.show()

## 7. Sample Responses Inspector

In [ ]:
# Compare GPT vs Gemini on the same prompt
def compare_responses(task, strategy, input_idx=1):
    subset = df[(df['Task']==task) & (df['Strategy']==strategy) & (df['Input_Index']==input_idx)]
    display(Markdown(f'### 🔍 {task} | {strategy} | Input {input_idx}'))
    for _, row in subset.iterrows():
        display(Markdown(f'**{row["Model"]}** ({row["Latency_sec"]}s | {row["Total_Tokens"]} tokens | ${row["Estimated_Cost_USD"]:.6f})'))
        display(Markdown(f'```\n{row["Response"]}\n```'))
        print()

# Task 1 — Log Triage
compare_responses('task1', 'chain_of_thought', 1)

In [ ]:
# Task 2 — Arabic PII
compare_responses('task2', 'few_shot', 1)

In [ ]:
# Task 3 — Complaint Reply
compare_responses('task3', 'chain_of_thought', 1)

## 8. Export Results

In [ ]:
os.makedirs('../outputs/raw', exist_ok=True)
os.makedirs('../outputs/processed', exist_ok=True)

# CSV
df.to_csv('../outputs/raw/results_raw.csv', index=False)
print('✅ Saved: outputs/raw/results_raw.csv')

# Excel with multiple sheets
with pd.ExcelWriter('../outputs/processed/scores.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='All Results', index=False)
    summary.to_excel(writer, sheet_name='Model Summary', index=False)
    
    # Per-task summaries
    for task in tasks:
        task_df = df[df['Task'] == task]
        task_summary = task_df.groupby(['Strategy', 'Model']).agg(
            Avg_Latency=('Latency_sec', 'mean'),
            Total_Cost=('Estimated_Cost_USD', 'sum'),
            Avg_Output_Len=('Output_Length', 'mean')
        ).round(4).reset_index()
        task_summary.to_excel(writer, sheet_name=task.capitalize(), index=False)

print('✅ Saved: outputs/processed/scores.xlsx (with multiple sheets)')

## 9. Generate Markdown Report

In [ ]:
# Determine winner per task
def get_task_winner(task):
    t = df[df['Task'] == task]
    cost = t.groupby('Model')['Estimated_Cost_USD'].sum()
    lat  = t.groupby('Model')['Latency_sec'].mean()
    return cost.idxmin(), lat.idxmin()

report_lines = [
    '# Prompt Stress-Test Suite — Results Report',
    '',
    '## Executive Summary',
    'Systematic evaluation of GPT-4o-mini vs Gemini 2.0 Flash across 3 business tasks, 3 prompting strategies, and 3 inputs each (54 total API calls).',
    '',
    '## Overall Model Comparison',
    '',
    summary.to_markdown(index=False),
    '',
    '## Per-Task Winners',
    '',
]

task_names = {
    'task1': 'Log Triage Summarizer',
    'task2': 'Arabic PII Redaction Checker',
    'task3': 'Customer Complaint Reply Drafter'
}

for task in tasks:
    cost_winner, lat_winner = get_task_winner(task)
    t = df[df['Task'] == task]
    task_summary = t.groupby(['Strategy', 'Model']).agg(
        Avg_Latency=('Latency_sec', 'mean'),
        Total_Cost=('Estimated_Cost_USD', 'sum')
    ).round(4).reset_index()
    
    report_lines += [
        f'### {task.upper()} — {task_names[task]}',
        f'- 💰 **Cost winner:** {cost_winner}',
        f'- ⚡ **Speed winner:** {lat_winner}',
        '',
        task_summary.to_markdown(index=False),
        ''
    ]

# Task 1 JSON reliability
report_lines += [
    '## Task 1 — JSON Output Reliability',
    '',
]
t1 = df[df['Task'] == 'task1']
if not t1.empty:
    json_success = t1.groupby(['Model', 'Strategy'])['Valid_JSON'].mean() * 100
    report_lines.append(json_success.reset_index().rename(columns={'Valid_JSON': 'Success_%'}).to_markdown(index=False))
    report_lines.append('')

report_lines += [
    '## Visualizations',
    '',
    '![Cost & Latency Comparison](outputs/charts/notebook_comparison.png)',
    '',
    '![Latency Heatmap](outputs/charts/latency_heatmap.png)',
]

report_content = '\n'.join(report_lines)

with open('../report.md', 'w', encoding='utf-8') as f:
    f.write(report_content)

print('✅ report.md generated!')
display(Markdown(report_content))

---
## ✅ All Done!

**Generated files:**
- `outputs/raw/results_raw.csv` — all 54 rows
- `outputs/processed/scores.xlsx` — multi-sheet scoring spreadsheet  
- `outputs/charts/notebook_comparison.png` — bar charts
- `outputs/charts/latency_heatmap.png` — heatmap
- `report.md` — full markdown report